In [1]:
import os, sys, time
sys.path.append(os.getenv("FALCON_PATH"))

In [ ]:
import json
from falcon import Falcon

model = Falcon(512)

In [3]:
def serialize_command(command: dict) -> bytes:
    return json.dumps(
        command,
        sort_keys=True,
        separators=(",", ":")
    ).encode()


In [4]:
class Drone:
    def __init__(self, drone_id):
        self.drone_id = drone_id
        self.secret_key, self.public_key = model.keygen()

    def sign_command(self, command: dict) -> bytes:
        serialized = serialize_command(command)
        return model.sign(self.secret_key, serialized)
    
    def verify_command(self, command: dict, signature: bytes) -> bool:
        serialized = serialize_command(command)
        model.verify(self.public_key, serialized, signature)
        return True


In [11]:
drone1 = Drone("DRONE_1")
drone2 = Drone("DRONE_2")
drone3 = Drone("DRONE_3")

In [12]:
command = {
    "command_id": "HOVER",
    "timestamp": 123456789,
    "payload": {"altitude": 10}
}

In [13]:
sig1 = drone1.sign_command(command)
sig2 = drone2.sign_command(command)
sig3 = drone3.sign_command(command)

print(len(sig1), len(sig2), len(sig3))
print("All signatures differ:",
      sig1 != sig2 and sig2 != sig3 and sig1 != sig3)

666 666 666
All signatures differ: True


In [14]:
drone1.verify_command(command, sig1)
drone2.verify_command(command, sig2)
drone3.verify_command(command, sig3)

print("ALL DRONES VERIFIED THEIR OWN SIGNATURES")

ALL DRONES VERIFIED THEIR OWN SIGNATURES
